In [0]:
import ast

dbutils.widgets.text("table_metadata", "")
table_metadata_str = dbutils.widgets.get("table_metadata")
if not table_metadata_str:
    raise ValueError("table_metadata cannot be empty")
table_metadata = ast.literal_eval(table_metadata_str)

dbutils.widgets.text("table_parameters", "")
table_parameters_str = dbutils.widgets.get("table_parameters")
if not table_parameters_str:
    raise ValueError("table_parameters cannot be empty")
table_parameters = ast.literal_eval(table_parameters_str)

dbutils.widgets.text("run_id", "")
run_id = dbutils.widgets.get("run_id")
if not run_id:
    import time
    run_id = str(int(time.time()))

print(f"run_id: {run_id}")


run_id: 1779798670


In [0]:
table_id = int(table_metadata.get("table_id"))
table_name = table_metadata.get("table_name")
bronze_schema = table_metadata.get("bronze_schema")
silver_schema = table_metadata.get("silver_schema")

load_type = table_parameters.get("load_type")
primary_key = table_parameters.get("primary_key")
watermark_column = table_parameters.get("watermark_column")

bronze_table = f"banking.{bronze_schema}.{table_name}"
silver_table = f"banking.{silver_schema}.{table_name}"

print(f"Table: {table_name}")
print(f"Load type: {load_type}")
print(f"Bronze: {bronze_table}")
print(f"Silver: {silver_table}")

Table: branches
Load type: FULL
Bronze: banking.bronze.branches
Silver: banking.silver.branches


In [0]:
status = "SUCCESS"
error_message = None
records_read = 0
records_written = 0

In [0]:
from pyspark.sql.functions import current_timestamp, col, max as spark_max, row_number, desc
from pyspark.sql.window import Window
from delta.tables import DeltaTable

try:
    # Read Bronze
    bronze_df = spark.table(bronze_table)

    # Filter by watermark
    if load_type in ["APPEND", "MERGE"] and watermark_column:
        watermark_df = spark.sql(f"""
            SELECT last_watermark_value
            FROM banking.metadata.table_watermarks
            WHERE table_id = {table_id}
        """)
        last_watermark = None
        if watermark_df.count() > 0:
            last_watermark = watermark_df.first()["last_watermark_value"]
        if last_watermark:
            bronze_df = bronze_df.filter(col(watermark_column).isNull() | (col(watermark_column) > last_watermark))

    # Deduplicate for MERGE
    if load_type == "MERGE":
        window = Window.partitionBy(primary_key).orderBy(desc(watermark_column))
        bronze_df = (
            bronze_df
            .withColumn("rn", row_number().over(window))
            .filter("rn = 1")
            .drop("rn")
        )

    records_read = bronze_df.count()
    print(f"Records to process: {records_read}")

    # Add audit columns
    bronze_df = (
        bronze_df
        .withColumn("insert_timestamp", current_timestamp())
        .withColumn("update_timestamp", current_timestamp())
    )

    # Create Silver schema
    spark.sql("CREATE SCHEMA IF NOT EXISTS banking.silver")

    # Write to Silver
    if not spark.catalog.tableExists(silver_table):
        bronze_df.write.format("delta").mode("overwrite").saveAsTable(silver_table)
        records_written = records_read
    else:
        if load_type == "FULL":
            bronze_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(silver_table)
            records_written = records_read
        elif load_type == "APPEND":
            bronze_df.write.format("delta").mode("append").saveAsTable(silver_table)
            records_written = records_read
        elif load_type == "MERGE":
            if not primary_key:
                raise ValueError("Primary key required for MERGE.")
            delta_table = DeltaTable.forName(spark, silver_table)
            (
                delta_table.alias("t")
                .merge(bronze_df.alias("s"), f"t.{primary_key} = s.{primary_key}")
                .whenMatchedUpdateAll()
                .whenNotMatchedInsertAll()
                .execute()
            )
            records_written = records_read
        else:
            raise ValueError("Unsupported load_type")

    # Update watermark
    if load_type in ["APPEND", "MERGE"] and watermark_column:
        max_value = bronze_df.agg(spark_max(col(watermark_column))).collect()[0][0]
        if max_value:
            spark.sql(f"""
                MERGE INTO banking.metadata.table_watermarks t
                USING (SELECT {table_id} AS table_id) s
                ON t.table_id = s.table_id
                WHEN MATCHED THEN UPDATE SET
                    last_watermark_value = '{max_value}',
                    last_updated_at = current_timestamp()
                WHEN NOT MATCHED THEN
                    INSERT (table_id, last_watermark_value, last_updated_at)
                    VALUES ({table_id}, '{max_value}', current_timestamp())
            """)

    print("Silver Load Completed Successfully.")

except Exception as e:
    status = "FAILED"
    error_message = str(e)
    print(f"Error: {error_message}")
    raise

finally:
    end_time = spark.sql("SELECT current_timestamp()").collect()[0][0]
    spark.sql(f"""
        UPDATE banking.metadata.pipeline_runs
        SET end_time = TIMESTAMP('{end_time}'),
            status = '{status}',
            number_of_records = {records_read},
            error_message = {'NULL' if not error_message else "'" + error_message.replace("'", "") + "'"}
        WHERE table_id = {table_id} AND run_id = {run_id}
    """)
    print("Audit record updated.")

Records to process: 4
Silver Load Completed Successfully.
Audit record updated.


In [0]:
%sql
SELECT COUNT(*) FROM banking.silver.branches

COUNT(*)
4


In [0]:
%sql
SELECT * FROM 
banking.metadata.pipeline_runs ORDER BY start_time DESC

run_id,table_id,layer,start_time,end_time,status,number_of_records,error_message
123,4,Bronze,2026-05-26T11:39:52.266Z,null,INPROGRESS,null,null
123,3,Bronze,2026-05-26T11:37:41.444Z,null,INPROGRESS,null,null
123,2,Bronze,2026-05-26T11:35:32.606Z,null,INPROGRESS,null,null
123,1,Bronze,2026-05-26T11:25:38.379Z,2026-05-26T11:46:34.170Z,SUCCESS,0,null
1234,5,Bronze,2026-05-26T05:05:14.937Z,2026-05-26T05:20:42.015Z,SUCCESS,5000,null
1234,6,Bronze,2026-05-25T12:30:19.729Z,2026-05-25T12:31:05.944Z,SUCCESS,5000,null
123,5,Bronze,2026-05-20T05:11:20.191Z,2026-05-21T12:08:06.120Z,SUCCESS,4000,null


In [0]:
%sql
SELECT 'customers' AS table_name, COUNT(*) AS cnt FROM banking.silver.customers
UNION ALL
SELECT 'accounts', COUNT(*) FROM banking.silver.accounts
UNION ALL
SELECT 'transactions', COUNT(*) FROM banking.silver.transactions
UNION ALL
SELECT 'branches', COUNT(*) FROM banking.silver.branches
UNION ALL
SELECT 'credit_bureau_reports', COUNT(*) FROM banking.silver.credit_bureau_reports
UNION ALL
SELECT 'payment_gateway_logs', COUNT(*) FROM banking.silver.payment_gateway_logs

table_name,cnt
customers,4000
accounts,4500
transactions,15000
branches,4
credit_bureau_reports,5000
payment_gateway_logs,20000
